# Paso 3.4 – Entrenamiento y selección de modelos
Se entrenan y comparan 4 modelos de clasificación clásica:
- Árbol de Decisión
- Naive Bayes
- KNN (K-Nearest Neighbors)
- SVM (Support Vector Machine)

Al final se exporta el mejor modelo en formato `.joblib` (paso 3.5).

## Celda 1 – Instalación de dependencias

In [1]:
# Descomenta si es necesario
# !pip install pandas scikit-learn joblib matplotlib seaborn

## Celda 2 – Importaciones

In [2]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

print("Bibliotecas cargadas correctamente.")

ModuleNotFoundError: No module named 'pandas'

## Celda 3 – Configuración
**Cambia `ARCHIVO_CSV` si tu archivo tiene otro nombre o ruta.**

In [ ]:
ARCHIVO_CSV      = "dataset.csv"           # ruta al CSV generado en el paso 3.3
TEST_SIZE        = 0.2                     # 80% entrenamiento, 20% prueba
RANDOM_STATE     = 42                      # semilla para reproducibilidad
N_COMPONENTS_PCA = 50                      # dimensiones para PCA (ayuda a KNN y SVM)

# Nombre del archivo de exportación (paso 3.5) — cámbialo con tus datos
NOMBRE_JOBLIB    = "carne_nombre_apellido.joblib"

## Celda 4 – Carga y partición del dataset

In [ ]:
df = pd.read_csv(ARCHIVO_CSV)

X = df.drop(columns=["etiqueta"]).values   # matriz de píxeles (n_muestras x 16384)
y = df["etiqueta"].values                  # vector de etiquetas

print(f"Dataset cargado: {X.shape[0]} muestras, {X.shape[1]} características")
print(f"Distribución de clases → positivo (1): {(y==1).sum()}  |  negativo (0): {(y==0).sum()}")

# Partición estratificada (mantiene proporción de clases en train y test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f"\nConjunto de entrenamiento : {X_train.shape[0]} muestras")
print(f"Conjunto de prueba        : {X_test.shape[0]} muestras")

## Celda 5 – Reducción de dimensionalidad con PCA
Con 16 384 características, KNN y SVM son muy lentos. PCA reduce las dimensiones conservando la mayor varianza posible. Árbol y Naive Bayes se entrenan sobre los datos originales.

In [ ]:
# Escalar antes de PCA (necesario para SVM y recomendado para KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Ajustar n_components al máximo posible si el dataset es pequeño
n_comp = min(N_COMPONENTS_PCA, X_train.shape[0] - 1, X_train.shape[1])

pca = PCA(n_components=n_comp, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca  = pca.transform(X_test_scaled)

varianza_acumulada = pca.explained_variance_ratio_.sum() * 100
print(f"PCA con {n_comp} componentes explica el {varianza_acumulada:.1f}% de la varianza")
print(f"Shape tras PCA: {X_train_pca.shape}")

## Celda 6 – Función auxiliar de evaluación

In [ ]:
resultados = {}   # guarda accuracy de cada modelo para comparar al final

def evaluar_modelo(nombre, modelo, X_tr, X_te, y_tr, y_te):
    """Entrena, predice e imprime métricas. Guarda accuracy en `resultados`."""
    modelo.fit(X_tr, y_tr)
    y_pred = modelo.predict(X_te)

    acc = accuracy_score(y_te, y_pred)
    resultados[nombre] = acc

    print(f"\n{'='*50}")
    print(f"  {nombre}")
    print(f"{'='*50}")
    print(f"  Accuracy : {acc:.4f} ({acc*100:.1f}%)")
    print()
    print(classification_report(y_te, y_pred,
                                target_names=["negativo (0)", "positivo (1)"]))

    # Matriz de confusión
    cm = confusion_matrix(y_te, y_pred)
    fig, ax = plt.subplots(figsize=(4, 3))
    ConfusionMatrixDisplay(cm, display_labels=["negativo", "positivo"]).plot(ax=ax, colorbar=False)
    ax.set_title(f"Matriz de confusión – {nombre}")
    plt.tight_layout()
    plt.show()

    return modelo

print("Función de evaluación definida.")

## Celda 7 – Modelo 1: Árbol de Decisión
Se usa `GridSearchCV` para probar distintas combinaciones de hiperparámetros y elegir la mejor.

In [ ]:
param_grid_dt = {
    "max_depth"        : [3, 5, 10, None],
    "min_samples_split": [2, 5, 10],
    "criterion"        : ["gini", "entropy"]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

grid_dt = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    param_grid_dt, cv=cv, scoring="accuracy", n_jobs=-1
)
grid_dt.fit(X_train, y_train)

print("Mejores hiperparámetros (Árbol):", grid_dt.best_params_)
mejor_dt = evaluar_modelo(
    "Árbol de Decisión",
    grid_dt.best_estimator_,
    X_train, X_test, y_train, y_test
)

## Celda 8 – Modelo 2: Naive Bayes

In [ ]:
# GaussianNB no tiene hiperparámetros críticos, se entrena directamente
mejor_nb = evaluar_modelo(
    "Naive Bayes",
    GaussianNB(),
    X_train, X_test, y_train, y_test
)

## Celda 9 – Modelo 3: KNN
Usa datos con PCA para acelerar el cálculo de distancias.

In [ ]:
param_grid_knn = {
    "n_neighbors": [3, 5, 7, 9],
    "metric"     : ["euclidean", "manhattan"]
}

grid_knn = GridSearchCV(
    KNeighborsClassifier(),
    param_grid_knn, cv=cv, scoring="accuracy", n_jobs=-1
)
grid_knn.fit(X_train_pca, y_train)

print("Mejores hiperparámetros (KNN):", grid_knn.best_params_)
mejor_knn = evaluar_modelo(
    "KNN",
    grid_knn.best_estimator_,
    X_train_pca, X_test_pca, y_train, y_test
)

## Celda 10 – Modelo 4: SVM
Usa datos con PCA. Se prueban kernel lineal y RBF.

In [ ]:
param_grid_svm = {
    "C"     : [0.1, 1, 10],
    "kernel": ["linear", "rbf"]
}

grid_svm = GridSearchCV(
    SVC(random_state=RANDOM_STATE),
    param_grid_svm, cv=cv, scoring="accuracy", n_jobs=-1
)
grid_svm.fit(X_train_pca, y_train)

print("Mejores hiperparámetros (SVM):", grid_svm.best_params_)
mejor_svm = evaluar_modelo(
    "SVM",
    grid_svm.best_estimator_,
    X_train_pca, X_test_pca, y_train, y_test
)

## Celda 11 – Comparación final de modelos

In [ ]:
print("\n" + "="*40)
print("  RESUMEN COMPARATIVO")
print("="*40)

for nombre, acc in sorted(resultados.items(), key=lambda x: x[1], reverse=True):
    barra = "█" * int(acc * 20)
    print(f"  {nombre:<25} {acc*100:5.1f}%  {barra}")

mejor_nombre = max(resultados, key=resultados.get)
print(f"\n  ✓ Mejor modelo: {mejor_nombre} ({resultados[mejor_nombre]*100:.1f}%)")

# Gráfico de barras
fig, ax = plt.subplots(figsize=(7, 4))
nombres = list(resultados.keys())
valores = [resultados[n] * 100 for n in nombres]
colores = ["#2ecc71" if n == mejor_nombre else "#3498db" for n in nombres]
bars = ax.bar(nombres, valores, color=colores, edgecolor="white", width=0.5)
ax.bar_label(bars, fmt="%.1f%%", padding=3)
ax.set_ylim(0, 115)
ax.set_ylabel("Accuracy (%)")
ax.set_title("Comparación de modelos – Accuracy en conjunto de prueba")
ax.axhline(y=100, color="gray", linestyle="--", linewidth=0.8)
plt.tight_layout()
plt.show()

## Celda 12 – Exportación del mejor modelo (Paso 3.5)
Se exporta en formato `.joblib`. **Cambia `NOMBRE_JOBLIB` en la Celda 3** con tu carné, nombre y apellido.

In [ ]:
# Seleccionar el objeto del mejor modelo
modelos_dict = {
    "Árbol de Decisión": mejor_dt,
    "Naive Bayes"      : mejor_nb,
    "KNN"              : mejor_knn,
    "SVM"              : mejor_svm,
}

modelo_a_exportar = modelos_dict[mejor_nombre]

# Para KNN y SVM necesitamos guardar también el scaler y el PCA
# Se empaqueta todo en un diccionario para que la inferencia sea reproducible
if mejor_nombre in ["KNN", "SVM"]:
    paquete = {
        "modelo" : modelo_a_exportar,
        "scaler" : scaler,
        "pca"    : pca,
        "tipo"   : mejor_nombre
    }
else:
    paquete = {
        "modelo" : modelo_a_exportar,
        "scaler" : None,
        "pca"    : None,
        "tipo"   : mejor_nombre
    }

joblib.dump(paquete, NOMBRE_JOBLIB)
print(f"Modelo exportado exitosamente: {NOMBRE_JOBLIB}")
print(f"Tipo: {mejor_nombre}  |  Accuracy: {resultados[mejor_nombre]*100:.1f}%")

## Celda 13 – Cómo usar el modelo exportado (inferencia)
Ejemplo de cómo cargar el `.joblib` para predecir una nueva imagen.

In [ ]:
# --- EJEMPLO DE INFERENCIA ---
# Cargar el modelo guardado
paquete_cargado = joblib.load(NOMBRE_JOBLIB)
modelo_cargado  = paquete_cargado["modelo"]

# Tomar una imagen del conjunto de prueba como ejemplo
muestra = X_test[0].reshape(1, -1)   # vector de 16384 píxeles

# Preprocesar si el modelo usa PCA
if paquete_cargado["scaler"] is not None:
    muestra = paquete_cargado["scaler"].transform(muestra)
    muestra = paquete_cargado["pca"].transform(muestra)

prediccion = modelo_cargado.predict(muestra)[0]
print(f"Predicción: {prediccion}  ({'POSITIVO – hay arroz' if prediccion == 1 else 'NEGATIVO – no hay arroz'})")
print(f"Etiqueta real : {y_test[0]}")